## **Proceso ETL**

#### **1. Importación de librerías**

In [2]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import time

#### **2. Configuración de la conexión**

In [3]:
USUARIO = "postgres"
PASSWORD = "1028"
HOST = "localhost"
PUERTO = "5432"
BASE_DATOS = "dw_analisis_criminalidad"

engine = create_engine(
    f"postgresql+psycopg2://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{BASE_DATOS}"
)

print("Conexión creada correctamente.")

Conexión creada correctamente.


Prueba de conexión

In [4]:
query = "SELECT * FROM dim_delito"

df = pd.read_sql(query, engine)
df.head()

,id_delito,tipo_delito


#### **4. Extracción de datos**

Nombre para columna tipo delito de acuerdo con nombre de archivo

In [11]:
TIPOS_DELITO = {
    "Reporte amenazas.csv": "Amenazas",
    "Reporte delitos sexuales.csv": "Delitos sexuales",
    "Reporte homicidios.csv": "Homicidio",
    "Reporte hurto motocicletas y automotores.csv": "Hurto de motocicletas y automotores",
    "Reporte hurto residencias y entidades comerciales.csv": "Hurto a residencias y entidades comerciales",
    "Reporte lesiones personales y accidente de transito.csv": "Lesiones personales",
    "Reporte violencia intrafamiliar.csv": "Violencia intrafamiliar"
}

Leer y transformar archivos

In [6]:
# Ruta de los archivos CSV
RUTA_DATOS_CSV = Path("Datos CSV")

In [12]:
archivos = sorted(RUTA_DATOS_CSV.glob("*.csv"))

print(f"Se encontraron {len(archivos)} archivos:\n")

for archivo in archivos:
    print(f"- {archivo.name}")

Se encontraron 7 archivos:

- Reporte amenazas.csv
- Reporte delitos sexuales.csv
- Reporte homicidios.csv
- Reporte hurto motocicletas y automotores.csv
- Reporte hurto residencias y entidades comerciales.csv
- Reporte lesiones personales y accidente de transito.csv
- Reporte violencia intrafamiliar.csv


In [17]:
dataframes = []

for archivo in archivos:

    print(f"Leyendo {archivo.name}...")

    df = pd.read_csv(
        archivo,
        encoding="utf-8-sig",
        low_memory=False
    )

    # Estandarizar nombres de columnas
    df.columns = (
        df.columns
          .str.strip()
          .str.upper()
    )

    # Corregir diferencias del archivo de homicidios
    if "GRUPO EDAD" in df.columns:

        df.rename(
            columns={
                "GRUPO EDAD": "GRUPO ETARIO"
            },
            inplace=True
        )

    # Eliminar columnas que no se utilizarán
    df.drop(
        columns=[
            "CODIGO DANE",
            "DELITOS",
            "DELITO",
            "TIPO DE HURTO",
            "DESCRIPCIÓN CONDUCTA"
        ],
        errors="ignore",
        inplace=True
    )

    # Crear nuestra columna DELITO
    df["DELITO"] = TIPOS_DELITO[archivo.name]

    dataframes.append(df)

    print(f"   Registros: {len(df):,}")

Leyendo Reporte amenazas.csv...
   Registros: 650,347
Leyendo Reporte delitos sexuales.csv...
   Registros: 392,576
Leyendo Reporte homicidios.csv...
   Registros: 154,433
Leyendo Reporte hurto motocicletas y automotores.csv...
   Registros: 641,663
Leyendo Reporte hurto residencias y entidades comerciales.csv...
   Registros: 633,804
Leyendo Reporte lesiones personales y accidente de transito.csv...
   Registros: 1,351,421
Leyendo Reporte violencia intrafamiliar.csv...
   Registros: 682,558


Unificar todos los dataframes

In [18]:
df = pd.concat(
    dataframes,
    ignore_index=True
)

Verificar resultado

In [19]:
print(f"\nTotal de registros: {len(df):,}")

print("\nColumnas:")

print(df.columns.tolist())


Total de registros: 4,506,802

Columnas:
['DEPARTAMENTO', 'MUNICIPIO', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD', 'DELITO']


Visualizar primeros registros

In [20]:
df.head()

,DEPARTAMENTO,MUNICIPIO,ARMAS MEDIOS,FECHA HECHO,GENERO,GRUPO ETARIO,CANTIDAD,DELITO
0,SUCRE,Coveñas,NO REPORTADO,22/06/2013,MASCULINO,ADULTOS,1.0,Amenazas
1,ATLÁNTICO,Barranquilla (CT),NO REPORTADO,13/12/2013,MASCULINO,ADULTOS,1.0,Amenazas
2,CAQUETÁ,Florencia (CT),NO REPORTADO,12/06/2012,FEMENINO,ADULTOS,1.0,Amenazas
3,VALLE,Cartago,NO REPORTADO,01/04/2012,MASCULINO,ADULTOS,1.0,Amenazas
4,VALLE,Cali (CT),NO REPORTADO,20/09/2012,FEMENINO,ADULTOS,1.0,Amenazas


#### **5. Transformación**

#### **6. Carga de dimensiones**

#### **7. Carga de la tabla de hechos**

#### **8. Validación**